# ML Classification Project
## Breast Cancer Prediction

**Goal**: Build and evaluate a supervised classification model to predict whether a breast tumor is malignant or benign.

**Requirements Covered**:
- Data preprocessing (scaling)
- Train/test split
- Cross-validation
- Comparing Logistic Regression and Random Forest
- Reporting metrics: Accuracy, Precision, Recall, F1, ROC-AUC
- Plots: ROC Curve, Confusion Matrix


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, roc_curve, 
                             confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')


### 1. Data Loading & Exploration
Using the built-in breast cancer dataset from `scikit-learn`.


In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(f"Dataset shape: {df.shape}")
display(df.head())
sns.countplot(x='target', data=df)
plt.title('Target Distribution')
plt.show()


### 2. Data Preprocessing & Train/Test Split
We split the data into training and testing sets, then scale the features using `StandardScaler` to ensure our Logistic Regression model performs optimally.


In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")


### 3. Model Training & Cross-Validation
We will compare two algorithms:
1. **Logistic Regression**
2. **Random Forest Classifier**


In [ ]:
log_reg = LogisticRegression(random_state=42)
rf_clf = RandomForestClassifier(random_state=42)
cv_log_reg = cross_val_score(log_reg, X_train_scaled, y_train, cv=5, scoring='accuracy')
cv_rf_clf = cross_val_score(rf_clf, X_train, y_train, cv=5, scoring='accuracy')
print(f"Logistic Regression CV Accuracy: {cv_log_reg.mean():.4f} +/- {cv_log_reg.std():.4f}")
print(f"Random Forest CV Accuracy: {cv_rf_clf.mean():.4f} +/- {cv_rf_clf.std():.4f}")
log_reg.fit(X_train_scaled, y_train)
rf_clf.fit(X_train, y_train)


### 4. Evaluation Metrics
We evaluate the trained models on the test set.


In [ ]:
def evaluate_model(model, X_t, y_t, model_name):
    y_pred = model.predict(X_t)
    y_prob = model.predict_proba(X_t)[:, 1]
    acc = accuracy_score(y_t, y_pred)
    prec = precision_score(y_t, y_pred)
    rec = recall_score(y_t, y_pred)
    f1 = f1_score(y_t, y_pred)
    roc_auc = roc_auc_score(y_t, y_prob)
    metrics = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC AUC': roc_auc
    }
    print(f"--- {model_name} Metrics ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    return metrics, y_pred, y_prob

print("Evaluating Logistic Regression (scaled data):")
metrics_lr, y_pred_lr, y_prob_lr = evaluate_model(log_reg, X_test_scaled, y_test, "Logistic Regression")
print("\nEvaluating Random Forest (unscaled data):")
metrics_rf, y_pred_rf, y_prob_rf = evaluate_model(rf_clf, X_test, y_test, "Random Forest")


### 5. Visualizations
- **ROC Curve** for comparing the trade-off between True Positive Rate and False Positive Rate.
- **Confusion Matrix** to analyze true positives, true negatives, false positives, and false negatives.


In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {metrics_lr['ROC AUC']:.4f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {metrics_rf['ROC AUC']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()
